# 04 — Splits: one train set, four test sets (LCO / LDO / LTO / LPO)

Each fold has **one train set** and **four test sets** representing different
generalization axes — new cell lines, new drugs, new tissues, and new pairs
(which itself mixes nothing-new / partially-new / fully-new pairs). This
replaces the earlier design of four independent train/test partitions — see
`SESSION_MEMORY.md` decision log for why.

1. Load data (same conventions as notebook 02/03)
2. Build the filtered `pairs` table and save it (index alignment source of truth)
3. Per fold: independently hold out 15% of cell lines / drugs / tissues
   (`GroupShuffleSplit`), build LCO/LDO/LTO test sets from whatever pairs touch
   each holdout (no purity requirement — a pair can land in more than one test
   set), build LPO test set as a random sample over **all** pairs sized at 2x
   the mean of the other three test sets, and the shared train set as
   everything else
4. Leakage assertions + composition diagnostics (how much of each test set is
   pure vs. compound)
5. Save four JSONs to `data/splits/*.json` — same file format as before, so
   downstream notebooks (07/08) don't need any changes to how they load splits

## Config

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data/GDSC2")
PROCESSED_DIR = Path("../data/processed")
SPLITS_DIR = Path("../data/splits")

COL_CELL_LINE = "cell_line_name"
COL_DRUG = "drug_name"
COL_IC50 = "LN_IC50"
COL_CELLOSAURUS = "cellosaurus_id"
COL_TISSUE = "tissue"

N_SPLITS = 5
RANDOM_STATE = 42

HOLDOUT_FRACTION = 0.15      # per axis: cell line, drug, tissue
LPO_SIZE_MULTIPLIER = 2.0    # LPO test size = this x mean(other three test set sizes)

In [ ]:
import json

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

## Load data (same three-way overlap logic as notebook 02/03)

In [3]:
rna = pd.read_csv(DATA_DIR / "gene_expression.csv", index_col=0)
protein = pd.read_csv(DATA_DIR / "proteomics.csv", index_col=0)
cell_lines = pd.read_csv(DATA_DIR / "cell_line_names.csv")
gdsc = pd.read_csv(DATA_DIR / "GDSC2.csv", low_memory=False)
drug_smiles = pd.read_csv(DATA_DIR / "drug_smiles.csv")

rna = rna[~rna.index.duplicated(keep="first")]
protein = protein[~protein.index.duplicated(keep="first")]


def three_way_overlap(
    rna_df: pd.DataFrame,
    protein_df: pd.DataFrame,
    gdsc_df: pd.DataFrame,
    mapping_df: pd.DataFrame,
) -> list[str]:
    """Cellosaurus IDs present in RNA, protein, and GDSC2 response data."""
    name_to_id = dict(zip(mapping_df[COL_CELL_LINE], mapping_df[COL_CELLOSAURUS]))
    gdsc_ids = set(gdsc_df[COL_CELL_LINE].map(name_to_id).dropna())
    return sorted(set(rna_df.index) & set(protein_df.index) & gdsc_ids)


common_ids = three_way_overlap(rna, protein, gdsc, cell_lines)
len(common_ids)

836

## Build the response pairs table

Filters to: cell line in the three-way overlap, drug has a SMILES string
(required for the GNN), and IC50 is not null. Index is reset to 0..n-1 —
this is the index space every split JSON below refers to.

In [4]:
def build_response_pairs(
    gdsc_df: pd.DataFrame,
    mapping_df: pd.DataFrame,
    drug_smiles_df: pd.DataFrame,
    common_ids: list[str],
) -> pd.DataFrame:
    name_to_id = dict(zip(mapping_df[COL_CELL_LINE], mapping_df[COL_CELLOSAURUS]))
    tissue_lookup = mapping_df.set_index(COL_CELLOSAURUS)[COL_TISSUE]

    pairs = gdsc_df.copy()
    pairs[COL_CELLOSAURUS] = pairs[COL_CELL_LINE].map(name_to_id)
    pairs = pairs[pairs[COL_CELLOSAURUS].isin(common_ids)]
    pairs = pairs[pairs[COL_DRUG].isin(drug_smiles_df[COL_DRUG])]
    pairs = pairs[pairs[COL_IC50].notna()]
    pairs[COL_TISSUE] = pairs[COL_CELLOSAURUS].map(tissue_lookup)

    return pairs.reset_index(drop=True)


pairs = build_response_pairs(gdsc, cell_lines, drug_smiles, common_ids)

n_pairs = len(pairs)
n_cells = pairs[COL_CELLOSAURUS].nunique()
n_drugs = pairs[COL_DRUG].nunique()
n_tissues = pairs[COL_TISSUE].nunique()
print(f"Response pairs after filtering: {n_pairs} "
      f"({n_cells} cell lines x {n_drugs} drugs x {n_tissues} tissues)")

Response pairs after filtering: 176197 (836 cell lines x 244 drugs x 26 tissues)


In [14]:
pairs

,cellosaurus_id,cell_line_name,pubchem_id,drug_name,Name,SignalQuality,pEC50_curvecurator,Slope,Front,Back,...,CELL_LINE_NAME,sample,drug,LN_IC50,AUC,IC50,min_dose_M,max_dose_M,LN_IC50_curvecurator,tissue
0,CVCL_0001,HEL,5352062,Romidepsin,CVCL_0001|5352062,0.0,8.089522,10.000000,1.002152,0.000100,...,HEL,CVCL_0001,5352062,-5.027970,0.892340,6.552098e-09,1.000530e-11,1.000000e-08,-4.810853,Blood
1,CVCL_0002,HL-60,36314,Paclitaxel,CVCL_0002|36314,0.0,8.396003,10.000000,1.273513,0.081264,...,HL-60,CVCL_0002,36314,-4.906881,0.875756,7.395519e-09,1.000530e-11,1.000000e-08,-5.455631,Blood
2,CVCL_0002,HL-60,5352062,Romidepsin,CVCL_0002|5352062,0.0,8.691853,1.212055,1.027599,0.000100,...,HL-60,CVCL_0002,5352062,-6.094023,0.755228,2.256313e-09,1.000530e-11,1.000000e-08,-6.153727,Blood
3,CVCL_0004,K-562,36314,Paclitaxel,CVCL_0004|36314,0.0,8.166454,2.553741,1.163487,0.358872,...,K-562,CVCL_0004,36314,-3.609454,0.940273,2.706662e-08,1.000530e-11,1.000000e-08,-4.382336,Blood
4,CVCL_0004,K-562,5352062,Romidepsin,CVCL_0004|5352062,0.0,8.326453,2.605769,0.958683,0.000100,...,K-562,CVCL_0004,5352062,-5.305005,0.825615,4.966673e-09,1.000530e-11,1.000000e-08,-5.389878,Blood
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176192,CVCL_0042,U2OS,12035,N-acetyl cysteine,CVCL_0042|12035,0.0,4.941199,10.000000,0.974500,0.949472,...,U-2-OS,CVCL_0042,12035.0,10.735991,0.984343,4.598134e-02,2.001054e-06,2.000000e-03,NaN,Bone
176193,CVCL_0320,HT-29,12035,N-acetyl cysteine,CVCL_0320|12035,0.0,3.450338,10.000000,0.924520,0.888075,...,HT-29,CVCL_0320,12035.0,10.243028,0.964711,2.808604e-02,2.001054e-06,2.000000e-03,NaN,Colon
176194,CVCL_0132,A-375,12035,N-acetyl cysteine,CVCL_0132|12035,0.0,5.197941,10.000000,1.018133,0.904789,...,A375,CVCL_0132,12035.0,9.961754,0.970325,2.119995e-02,2.001054e-06,2.000000e-03,NaN,Skin
176195,CVCL_0547,SW620,12035,N-acetyl cysteine,CVCL_0547|12035,0.0,4.799151,10.000000,0.923847,0.875740,...,SW620,CVCL_0547,12035.0,9.959085,0.948461,2.114344e-02,2.001054e-06,2.000000e-03,NaN,Colon


## Save the pairs table

Without this, the split JSON indices below are meaningless later — this is
the reproducibility link between splits and the actual filtered data.

In [5]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
pairs.to_parquet(PROCESSED_DIR / "response_pairs.parquet")

## Split generators

### Step 1 — per-axis holdout groups

15% of cell lines, 15% of drugs, 15% of tissues, drawn independently per fold
(`GroupShuffleSplit`, not a strict partition — draws can overlap across folds,
which is fine since each fold is evaluated independently). Different
`random_state` offset per axis so the three draws aren't correlated through
identical seeding.

**Deviation flagged:** the old LCO split used `StratifiedGroupKFold` to keep
tissue composition balanced across folds. Dropped here — tissue is now its own
holdout axis (LTO), and the composition diagnostics below report how much of
each LCO test set also happens to touch a held-out tissue, so the information
isn't lost, just surfaced differently. Revisit if tissue drift in LCO test
composition turns out to matter.

In [ ]:
def make_axis_holdouts(group_values: pd.Series, n_splits: int, fraction: float, seed: int) -> list[set]:
    """n_splits independent ~`fraction`-sized random draws of unique group values.
    Not a strict partition: draws can overlap across folds."""
    n_groups = group_values.nunique()
    min_required = max(1, round(1 / fraction))
    if n_groups < min_required:
        raise ValueError(
            f"Cannot draw {fraction:.0%} holdouts from only {n_groups} unique values "
            f"in '{group_values.name}'."
        )
    gss = GroupShuffleSplit(n_splits=n_splits, test_size=fraction, random_state=seed)
    idx = np.arange(len(group_values))
    return [set(group_values.iloc[test_idx]) for _, test_idx in gss.split(idx, groups=group_values)]


print(f"Unique cell lines: {pairs[COL_CELLOSAURUS].nunique()}")
print(f"Unique drugs:      {pairs[COL_DRUG].nunique()}")
print(f"Unique tissues:    {pairs[COL_TISSUE].nunique()}  <- tightest axis, check this isn't too small for 15%")

cell_holdouts = make_axis_holdouts(pairs[COL_CELLOSAURUS], N_SPLITS, HOLDOUT_FRACTION, RANDOM_STATE)
drug_holdouts = make_axis_holdouts(pairs[COL_DRUG], N_SPLITS, HOLDOUT_FRACTION, RANDOM_STATE + 1)
tissue_holdouts = make_axis_holdouts(pairs[COL_TISSUE], N_SPLITS, HOLDOUT_FRACTION, RANDOM_STATE + 2)

print(f"Cell lines held out per fold: {[len(s) for s in cell_holdouts]} (of {pairs[COL_CELLOSAURUS].nunique()})")
print(f"Drugs held out per fold:      {[len(s) for s in drug_holdouts]} (of {pairs[COL_DRUG].nunique()})")
print(f"Tissues held out per fold:    {[len(s) for s in tissue_holdouts]} (of {pairs[COL_TISSUE].nunique()})")

### Step 2 — build train + four test sets per fold

`LCO_test` / `LDO_test` / `LTO_test` = every pair touching that axis's
held-out group, regardless of the other two axes (no purity rule). `LPO_test`
is a random sample over **all** pairs, sized at `LPO_SIZE_MULTIPLIER` x the
mean size of the other three test sets, so it naturally mixes nothing-new /
partially-new / fully-new pairs. `train` = pairs touching none of the three
held-out groups, minus whatever of those got randomly pulled into `LPO_test`
— **the same train set feeds all four test sets in a given fold.**

In [ ]:
def build_fold(
    pairs_df: pd.DataFrame,
    cell_holdout: set,
    drug_holdout: set,
    tissue_holdout: set,
    lpo_multiplier: float,
    seed: int,
) -> dict:
    """One shared train set + four test sets for a single fold."""
    idx_all = pairs_df.index.to_numpy()
    is_new_cell = pairs_df[COL_CELLOSAURUS].isin(cell_holdout).to_numpy()
    is_new_drug = pairs_df[COL_DRUG].isin(drug_holdout).to_numpy()
    is_new_tissue = pairs_df[COL_TISSUE].isin(tissue_holdout).to_numpy()

    lco_test = idx_all[is_new_cell]
    ldo_test = idx_all[is_new_drug]
    lto_test = idx_all[is_new_tissue]

    untouched = idx_all[~is_new_cell & ~is_new_drug & ~is_new_tissue]

    target_lpo_n = round(lpo_multiplier * np.mean([len(lco_test), len(ldo_test), len(lto_test)]))
    rng = np.random.default_rng(seed)
    lpo_test = rng.choice(idx_all, size=target_lpo_n, replace=False)

    train = np.setdiff1d(untouched, lpo_test, assume_unique=False)

    return {
        "train": train, "lco_test": lco_test, "ldo_test": ldo_test,
        "lto_test": lto_test, "lpo_test": lpo_test,
    }


folds = [
    build_fold(
        pairs, cell_holdouts[i], drug_holdouts[i], tissue_holdouts[i],
        LPO_SIZE_MULTIPLIER, RANDOM_STATE + 100 + i,
    )
    for i in range(N_SPLITS)
]

for i, f in enumerate(folds):
    print(f"fold {i}: train={len(f['train']):,} | lco_test={len(f['lco_test']):,} "
          f"| ldo_test={len(f['ldo_test']):,} | lto_test={len(f['lto_test']):,} "
          f"| lpo_test={len(f['lpo_test']):,}")

### Leakage assertions

LCO/LDO/LTO: the held-out group must never appear in train — train is built
from the untouched pool by construction, but assert it explicitly anyway.
LPO: no row-index overlap with train (group-level overlap with train *is*
expected and fine for LPO — that's the point of sampling from all pairs).

In [ ]:
def assert_no_group_leakage(pairs_df: pd.DataFrame, train_idx: np.ndarray, test_idx: np.ndarray, group_col: str) -> None:
    train_groups = set(pairs_df.loc[train_idx, group_col])
    test_groups = set(pairs_df.loc[test_idx, group_col])
    overlap = train_groups & test_groups
    assert not overlap, f"{group_col} leakage: {overlap}"


def assert_no_row_overlap(train_idx: np.ndarray, test_idx: np.ndarray) -> None:
    overlap = set(train_idx) & set(test_idx)
    assert not overlap, f"{len(overlap)} rows appear in both train and test"


for i, f in enumerate(folds):
    assert_no_group_leakage(pairs, f["train"], f["lco_test"], COL_CELLOSAURUS)
    assert_no_group_leakage(pairs, f["train"], f["ldo_test"], COL_DRUG)
    assert_no_group_leakage(pairs, f["train"], f["lto_test"], COL_TISSUE)
    assert_no_row_overlap(f["train"], f["lpo_test"])

print("All leakage assertions passed.")

## Composition diagnostics

For LCO/LDO/LTO: how much of each test set is **pure** (only that one axis is
new) vs. **compound** (also touches another held-out axis) — tells you how
confounded a drop in that metric is. For LPO: breakdown by how many axes are
new (0 = nothing new, 1 = one axis new, 2-3 = compound) — this *is* LPO's
intended composition under this design, not a confound to explain away.

In [ ]:
def novelty_flags(pairs_df: pd.DataFrame, idx: np.ndarray, cell_holdout: set, drug_holdout: set, tissue_holdout: set) -> pd.DataFrame:
    sub = pairs_df.loc[idx]
    return pd.DataFrame({
        "new_cell": sub[COL_CELLOSAURUS].isin(cell_holdout).to_numpy(),
        "new_drug": sub[COL_DRUG].isin(drug_holdout).to_numpy(),
        "new_tissue": sub[COL_TISSUE].isin(tissue_holdout).to_numpy(),
    })


def summarize_fold(pairs_df: pd.DataFrame, fold_i: int, fold: dict, cell_holdout: set, drug_holdout: set, tissue_holdout: set) -> pd.DataFrame:
    rows = []
    for label, test_idx in [
        ("LCO", fold["lco_test"]), ("LDO", fold["ldo_test"]),
        ("LTO", fold["lto_test"]), ("LPO", fold["lpo_test"]),
    ]:
        flags = novelty_flags(pairs_df, test_idx, cell_holdout, drug_holdout, tissue_holdout)
        n_new = flags[["new_cell", "new_drug", "new_tissue"]].sum(axis=1)
        rows.append({
            "fold": fold_i, "test_set": label, "n": len(test_idx),
            "nothing_new_frac": round((n_new == 0).mean(), 3) if len(test_idx) else float("nan"),
            "pure_frac": round((n_new == 1).mean(), 3) if len(test_idx) else float("nan"),
            "compound_frac": round((n_new >= 2).mean(), 3) if len(test_idx) else float("nan"),
        })
    return pd.DataFrame(rows)


composition = pd.concat([
    summarize_fold(pairs, i, f, cell_holdouts[i], drug_holdouts[i], tissue_holdouts[i])
    for i, f in enumerate(folds)
], ignore_index=True)
composition

## Save all four splits

In [ ]:
lpo = [{"train": f["train"].tolist(), "test": f["lpo_test"].tolist()} for f in folds]
lco = [{"train": f["train"].tolist(), "test": f["lco_test"].tolist()} for f in folds]
ldo = [{"train": f["train"].tolist(), "test": f["ldo_test"].tolist()} for f in folds]
lto = [{"train": f["train"].tolist(), "test": f["lto_test"].tolist()} for f in folds]


def save_splits(splits: list[dict], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as fh:
        json.dump(splits, fh, separators=(",", ":"))  # compact, no indent


for name, splits in [("lpo", lpo), ("lco", lco), ("ldo", ldo), ("lto", lto)]:
    save_splits(splits, SPLITS_DIR / f"{name}.json")
    fold_sizes = [(len(s["train"]), len(s["test"])) for s in splits]
    print(f"{name.upper()}: {fold_sizes}")

## Save per-fold holdout group identities

The split JSONs only store row indices — they don't say *which* cell lines, drugs, or tissues were actually held out. Needed for interpretation later (e.g. attributing a fold's LTO performance to which specific tissues landed in it, or checking if certain drugs/cell lines are consistently hard across folds).

In [ ]:
holdout_groups = [
    {
        "fold": i,
        "cell_lines_held_out": sorted(cell_holdouts[i]),
        "drugs_held_out": sorted(drug_holdouts[i]),
        "tissues_held_out": sorted(tissue_holdouts[i]),
    }
    for i in range(N_SPLITS)
]

SPLITS_DIR.mkdir(parents=True, exist_ok=True)
with open(SPLITS_DIR / "holdout_groups.json", "w") as fh:
    json.dump(holdout_groups, fh, indent=2)

for g in holdout_groups:
    print(f"fold {g['fold']}: "
          f"{len(g['cell_lines_held_out'])} cell lines, "
          f"{len(g['drugs_held_out'])} drugs, "
          f"{len(g['tissues_held_out'])} tissues -> {g['tissues_held_out']}")